In [1]:
%pip install --upgrade transformers accelerate peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.2/562.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# # 코드를 단일 GPU에서만 실행되도록 설정
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # 첫 번째 GPU만 사용
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
import os
import gc
import torch
import datasets
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

2025-09-16 05:18:00.326664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757999880.664371      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757999880.760370      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
# 해당 블록만 자기 환경에 맞게 조정하면 그냥 쭉 실행 가능합니당
HF_TOKEN = "REDACTED_HF_TOKEN"
login(token=HF_TOKEN)

# 사용할 모델 이름
BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
# 데이터 및 모델을 저장할 로컬 경로
PROCESSED_DATA_PATH = "./data"
FINAL_MODEL_PATH = "./finetuned_model"

### 1. data load, preprocess, tokenize

In [5]:
print("Step 1: 데이터셋 로딩 및 분할...")

dataset = load_dataset('FinGPT/fingpt-fiqa_qa')
dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)

print("\nStep 2: 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, token=HF_TOKEN)

# Llama-3는 pad 토큰이 없어서 eos 토큰을 pad 토큰으로 사용해야댐
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_seq_length = 512

Step 1: 데이터셋 로딩 및 분할...


README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001-ab79bf9300210e(…):   0%|          | 0.00/10.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17110 [00:00<?, ? examples/s]


Step 2: 토크나이저 로드


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [6]:
def transform_and_tokenize(example):
    context = (f"Instruction: {example['instruction']}\nInput: {example['input']}\nAnswer: ")
    output = example['output']
    
    prompt_ids = tokenizer.encode(context, max_length=max_seq_length, truncation=True)
    target_ids = tokenizer.encode(output, max_length=max_seq_length, truncation=True, add_special_tokens=False)
    
    input_ids = prompt_ids + target_ids + [tokenizer.eos_token_id]
    
    return {
        'input_ids': input_ids,
        'seq_len': len(prompt_ids)
    }

In [7]:
# .map()을 사용하여 전체 데이터셋에 함수 적용
tokenized_dataset = dataset.map(transform_and_tokenize)

# print(f"\nStep 4: 처리된 데이터셋을 로컬에 저장: '{PROCESSED_DATA_PATH}'")
# tokenized_dataset.save_to_disk(PROCESSED_DATA_PATH)

Map:   0%|          | 0/15399 [00:00<?, ? examples/s]

Map:   0%|          | 0/1711 [00:00<?, ? examples/s]

### 2. model load, 양자화, LoRA 설정

In [8]:
print("\nStep 5: 모델 로딩 및 양자화 설정...")
# 1. BitsAndBytesConfig 객체 생성
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. from_pretrained 호출 시 quantization_config 인자에 전달
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    token = HF_TOKEN,
    quantization_config=bnb_config, 
    device_map="auto",
)

tokenizer.padding_side = 'left'


Step 5: 모델 로딩 및 양자화 설정...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [9]:
print("\nStep 6: LoRA 설정 적용...")
# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], # Llama-3의 일반적인 타겟 모듈
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)


Step 6: LoRA 설정 적용...


In [10]:
# PEFT 모델로 변환
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


### 4. Train

In [11]:
# print(f"\nStep 7: 저장된 토크나이징 데이터 로드: '{PROCESSED_DATA_PATH}'")
# 저장된 데이터셋 다시 불러오기
# tokenized_dataset = datasets.load_from_disk(PROCESSED_DATA_PATH)

In [12]:
def data_collator(features: list) -> dict:
    len_ids = [len(feature["input_ids"]) for feature in features]
    longest = max(len_ids)
    input_ids = []
    labels_list = []
    
    for ids_l, feature in sorted(zip(len_ids, features), key=lambda x: -x[0]):
        ids = feature["input_ids"]
        seq_len = feature["seq_len"]
        labels = ([-100] * (seq_len - 1) + ids[(seq_len - 1):])
        
        padding_length = longest - ids_l
        ids = ([tokenizer.pad_token_id] * padding_length) + ids
        labels = ([-100] * padding_length) + labels
        input_ids.append(torch.LongTensor(ids))
        labels_list.append(torch.LongTensor(labels))

            
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels_list),
    }

In [13]:
print("\nStep 8: 학습 인자(TrainingArguments) 설정...")
training_args = TrainingArguments(
    output_dir=FINAL_MODEL_PATH,
    label_names=["labels"],
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False
)


Step 8: 학습 인자(TrainingArguments) 설정...


In [14]:
print("\nStep 9: Trainer 초기화 및 학습 시작...")
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

trainer.train()


Step 9: Trainer 초기화 및 학습 시작...


Step,Training Loss,Validation Loss
200,2.454700,2.484642
400,2.435800,2.462928


TrainOutput(global_step=482, training_loss=2.48304137550449, metrics={'train_runtime': 21473.0533, 'train_samples_per_second': 0.717, 'train_steps_per_second': 0.022, 'total_flos': 2.290215168735068e+17, 'train_loss': 2.48304137550449, 'epoch': 1.0})

In [15]:
print(f"\nStep 10: 최종 모델 저장: '{FINAL_MODEL_PATH}'")
model.save_pretrained(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)


Step 10: 최종 모델 저장: './finetuned_model'


('./finetuned_model/tokenizer_config.json',
 './finetuned_model/special_tokens_map.json',
 './finetuned_model/chat_template.jinja',
 './finetuned_model/tokenizer.json')